In [9]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "master").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "master" / "data"
OUTPUT_DIR = DATA_DIR / "knowledge" / "outputs"

from master.data.knowledge.extract_pipeline import (
    ExtractionConfig,
    chunk_markdown_corpus,
    export_graph_json,
    list_markdown_files,
    merge_graph_documents,
)
from master.data.knowledge.fpt_client import (
    FPTChatConfig,
    extract_graph_for_chunk_fpt,
)

In [2]:
markdown_files = list_markdown_files(DATA_DIR)
print(f"Found {len(markdown_files)} markdown textbooks")
for path in markdown_files:
    print("-", path.name)

Found 6 markdown textbooks
- sach-giao-khoa-toan-10-tap-1-ket-noi-tri-thuc-voi-cuoc-song.md
- sach-giao-khoa-toan-10-tap-2-ket-noi-tri-thuc-voi-cuoc-song.md
- sach-giao-khoa-toan-11-tap-1-ket-noi-tri-thuc-voi-cuoc-song.md
- sach-giao-khoa-toan-11-tap-2-ket-noi-tri-thuc-voi-cuoc-song.md
- sach-giao-khoa-toan-12-tap-1-ket-noi-tri-thuc-voi-cuoc-song.md
- sach-giao-khoa-toan-12-tap-2-ket-noi-tri-thuc-voi-cuoc-song.md


In [3]:
config = ExtractionConfig(
    min_body_chars=180,
    max_chunk_chars=6000,
    keep_exercise_sections=False,
)

chunks = chunk_markdown_corpus(markdown_files, config=config)
print(f"Built {len(chunks)} candidate chunks")
preview = [
    {
        "chunk_id": chunk.chunk_id,
        "title": chunk.title,
        "chapter_title": chunk.chapter_title,
        "chars": len(chunk.content),
    }
    for chunk in chunks[:12]
]
preview

Built 261 candidate chunks


[{'chunk_id': 'toan-10-tap-1:0001',
  'title': '1. MÊNH ĐỂ, MÊNH ĐỂ CHỰA BIẾN',
  'chapter_title': None,
  'chars': 2738},
 {'chunk_id': 'toan-10-tap-1:0002',
  'title': '2. MỆNH ĐỂ PHỦ ĐỊNH',
  'chapter_title': None,
  'chars': 1177},
 {'chunk_id': 'toan-10-tap-1:0003',
  'title': '3. MỆNH ĐỀ KÉO THEO, MỆNH ĐỀ ĐẢO',
  'chapter_title': None,
  'chars': 2237},
 {'chunk_id': 'toan-10-tap-1:0004',
  'title': '4. MỆNH ĐỀ TƯƠNG ĐƯƠNG',
  'chapter_title': None,
  'chars': 1216},
 {'chunk_id': 'toan-10-tap-1:0005',
  'title': '5. MỆNH ĐỀ CÓ CHỨA KÍ HIỆU ¥, 3',
  'chapter_title': None,
  'chars': 1834},
 {'chunk_id': 'toan-10-tap-1:0006',
  'title': 'TẬP HỢP VÀ CÁC PHÉP TOÁN TRÊN TẬP HƠP',
  'chapter_title': None,
  'chars': 699},
 {'chunk_id': 'toan-10-tap-1:0007',
  'title': '1. CÁC KHÁI NIỆM CƠ BẢN VỀ TẬP HỢP',
  'chapter_title': None,
  'chars': 3797},
 {'chunk_id': 'toan-10-tap-1:0008',
  'title': '2. CÁC TẬP HỢP SỐ',
  'chapter_title': None,
  'chars': 5920},
 {'chunk_id': 'toan-10-tap-1

In [4]:
sample_chunk = chunks[0]
print(sample_chunk.chunk_id)
print(sample_chunk.title)
print(sample_chunk.chapter_title)
print("=" * 80)
print(sample_chunk.combined_text[:2500])

toan-10-tap-1:0001
1. MÊNH ĐỂ, MÊNH ĐỂ CHỰA BIẾN
None
Chunk ID: toan-10-tap-1:0001
Book ID: toan-10-tap-1
Grade: 10
Title: 1. MÊNH ĐỂ, MÊNH ĐỂ CHỰA BIẾN

### a. Mênh để

- 🍞 🚻 Trong các câu ở tình huống mở đầu:
  - a) Câu nào đúng?
  - b) Câu nào sai?
  - c) Câu nào không xác định được tính đúng sai?

Những câu nói của An và Khoa là những khẳng định có tính đúng hoặc sai. Người ta gọi mỗi câu như vậy là một mệnh để lôgic (gọi tắt là mệnh để). Những câu không xác định được tính đúng sai không phải là mệnh để.

Mỗi mệnh đề phải hoặc đúng hoặc sai.

Một mệnh đề không thể vừa đúng vừa sai.

Chú ý. Người ta thường sử dụng các chữ cái P, Q, R,... để biểu thị các mệnh đề.

- >> Ví dụ 1. Trong các câu sau, câu nào là mệnh đề? Câu nào không phải là mệnh đề?
  - a) Phương trình  $3x^2 5x + 2 = 0$  có nghiệm nguyên;
  - b) 5 < 7 3:
  - c) Có bao nhiều dấu hiệu nhận biết hai tam giác đồng dạng?
  - d) Đấy là cách xử lí khôn ngoan!

Thông thường, những câu nghi vấn, câu cảm thán, câu cầu khiến khôn

In [5]:
import importlib
import os

if importlib.util.find_spec("dotenv") is not None:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")

if importlib.util.find_spec("pydantic") is None:
    raise RuntimeError(
        "Missing dependency: pydantic. Install it before running extraction cells."
    )

fpt_config = FPTChatConfig.from_env()

if not fpt_config.api_key:
    raise RuntimeError("Missing FPT_API_KEY in environment or .env file.")

Using FPT base URL: https://mkp-api.fptcloud.com


In [ ]:
sample_document = extract_graph_for_chunk_fpt(sample_chunk, config=fpt_config)
sample_document.model_dump(mode="json")

AttributeError: 'list' object has no attribute 'combined_text'

In [11]:
import importlib
import master.data.knowledge.fpt_client as fpt_client
importlib.reload(fpt_client)

<module 'master.data.knowledge.fpt_client' from '/home/phuckhang/MyWorkspace/GDGoC_HackathonVietnam/master/data/knowledge/fpt_client.py'>

In [12]:
RUN_LIMIT = 10
# Set RUN_LIMIT = None after the sample output looks stable.

selected_chunks = chunks if RUN_LIMIT is None else chunks[:RUN_LIMIT]
documents = [extract_graph_for_chunk_fpt(chunk, config=fpt_config) for chunk in selected_chunks]
merged_graph = merge_graph_documents(documents)

suffix = "all" if RUN_LIMIT is None else str(RUN_LIMIT)
merged_path = export_graph_json(merged_graph, OUTPUT_DIR / f"merged_graph_{suffix}.json")
chunk_docs_path = OUTPUT_DIR / f"chunk_documents_{suffix}.json"
chunk_docs_path.parent.mkdir(parents=True, exist_ok=True)
chunk_docs_path.write_text(
    json.dumps([doc.model_dump(mode="json") for doc in documents], ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Saved merged graph to: {merged_path}")
print(f"Saved chunk documents to: {chunk_docs_path}")
print(f"Merged nodes: {len(merged_graph.nodes)}")
print(f"Merged edges: {len(merged_graph.edges)}")

Saved merged graph to: /home/phuckhang/MyWorkspace/GDGoC_HackathonVietnam/master/data/knowledge/outputs/merged_graph_10.json
Saved chunk documents to: /home/phuckhang/MyWorkspace/GDGoC_HackathonVietnam/master/data/knowledge/outputs/chunk_documents_10.json
Merged nodes: 67
Merged edges: 30


In [14]:
from collections import Counter

# 1) Lấy toàn bộ file markdown
markdown_files = list_markdown_files(DATA_DIR)
print(f"Found {len(markdown_files)} markdown textbooks")
for path in markdown_files:
    print("-", path.name)

# 2) Chunk toàn bộ corpus
cfg = ExtractionConfig()
chunks = chunk_markdown_corpus(markdown_files, config=cfg)
print(f"\nTotal chunks: {len(chunks)}")

book_counter = Counter(chunk.book_id for chunk in chunks)
print("Chunks by book:")
for book_id, count in sorted(book_counter.items()):
    print(f"- {book_id}: {count}")

# 3) Extract toàn bộ knowledge graph theo từng chunk
documents = []
errors = []

for idx, chunk in enumerate(chunks, start=1):
    print(f"[{idx}/{len(chunks)}] Extracting {chunk.chunk_id} | {chunk.title}")
    try:
        doc = extract_graph_for_chunk_fpt(chunk, config=fpt_config)
        documents.append(doc)
    except Exception as exc:
        errors.append(
            {
                "chunk_id": chunk.chunk_id,
                "book_id": chunk.book_id,
                "title": chunk.title,
                "source_path": chunk.source_path,
                "error": str(exc),
            }
        )
        print(f"  -> ERROR: {exc}")

print(f"\nSuccessful documents: {len(documents)}")
print(f"Failed chunks: {len(errors)}")

if not documents:
    raise RuntimeError("Không extract được chunk nào, dừng để tránh merge graph rỗng.")

# 4) Merge graph toàn bộ
merged_graph = merge_graph_documents(documents)

# 5) Lưu kết quả
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

merged_path = export_graph_json(merged_graph, OUTPUT_DIR / "merged_graph_all.json")

chunk_docs_path = OUTPUT_DIR / "chunk_documents_all.json"
chunk_docs_path.write_text(
    json.dumps(
        [doc.model_dump(mode="json") for doc in documents],
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

errors_path = OUTPUT_DIR / "extraction_errors_all.json"
errors_path.write_text(
    json.dumps(errors, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# 6) In summary
print("\nSaved files:")
print(f"- merged graph: {merged_path}")
print(f"- chunk docs:   {chunk_docs_path}")
print(f"- errors:       {errors_path}")

print("\nMerged graph stats:")
print(f"- nodes: {len(merged_graph.nodes)}")
print(f"- edges: {len(merged_graph.edges)}")
print(f"- successful chunks: {len(documents)}")
print(f"- failed chunks: {len(errors)}")


Found 6 markdown textbooks
- sach-giao-khoa-toan-10-tap-1-ket-noi-tri-thuc-voi-cuoc-song.md
- sach-giao-khoa-toan-10-tap-2-ket-noi-tri-thuc-voi-cuoc-song.md
- sach-giao-khoa-toan-11-tap-1-ket-noi-tri-thuc-voi-cuoc-song.md
- sach-giao-khoa-toan-11-tap-2-ket-noi-tri-thuc-voi-cuoc-song.md
- sach-giao-khoa-toan-12-tap-1-ket-noi-tri-thuc-voi-cuoc-song.md
- sach-giao-khoa-toan-12-tap-2-ket-noi-tri-thuc-voi-cuoc-song.md

Total chunks: 261
Chunks by book:
- toan-10-tap-1: 45
- toan-10-tap-2: 63
- toan-11-tap-1: 71
- toan-11-tap-2: 32
- toan-12-tap-1: 25
- toan-12-tap-2: 25
[1/261] Extracting toan-10-tap-1:0001 | 1. MÊNH ĐỂ, MÊNH ĐỂ CHỰA BIẾN
[2/261] Extracting toan-10-tap-1:0002 | 2. MỆNH ĐỂ PHỦ ĐỊNH
[3/261] Extracting toan-10-tap-1:0003 | 3. MỆNH ĐỀ KÉO THEO, MỆNH ĐỀ ĐẢO
[4/261] Extracting toan-10-tap-1:0004 | 4. MỆNH ĐỀ TƯƠNG ĐƯƠNG
[5/261] Extracting toan-10-tap-1:0005 | 5. MỆNH ĐỀ CÓ CHỨA KÍ HIỆU ¥, 3
[6/261] Extracting toan-10-tap-1:0006 | TẬP HỢP VÀ CÁC PHÉP TOÁN TRÊN TẬP HƠP
[7/261] Extr